# 8ФМ AI — обучение модели `lu_sol` на Kaggle

Этот ноутбук обучает **максимальную** Classic-политику проекта — `lu_sol` —
на мощностях Kaggle, а не на ноутбуке.

Что делает `lu_sol` по сравнению с `lu_luna`:

* **больше данных в начальном векторе** — расклад признаков `v2`
  (158 входов состояния / 32 входа действия вместо 115 / 25): прогресс
  комбо, экономика закрытых карт, контекст хода, давление по столу,
  давление Alexey и близость к порогу 60 очков;
* **шире сеть** — 512/256 в башне состояния и 256 в башне действий;
* **дольше обучение** — 6000 апдейтов вместо 320, длинный BC-разогрев,
  больше PPO-эпох и батч;
* **крупнее лига** самопгры и шире периодическая оценка.

Обучение **разбито на чанки по времени** и возобновляется между сессиями:
одна сессия Kaggle (~12 ч) не вмещает весь прогон, но следующий запуск
продолжает ровно с места остановки.

## Настройки Kaggle (обязательно)

1. Справа в панели **Settings** включите **Internet: On** — без этого `git clone` не сработает.
2. **Accelerator: None**. GPU здесь не нужен и не поможет: симулятор и сеть
   написаны на чистом Python + NumPy, узкое место — проигрыш игр, а его
   ускоряет только параллельный сэмплинг на ядрах CPU (`--workers 4`).
3. Нажмите **Save Version → Save & Run All (Commit)**, иначе сессия завершится
   сразу после закрытия вкладки. На выполнение даётся до ~12 часов.

In [ ]:
import os

REPO_URL = "https://github.com/leonka123/8FM_AI"
WORK_DIR = "/kaggle/working"

if not os.path.isdir("8FM_AI"):
    !git clone --depth 1 {REPO_URL} 8FM_AI
%cd 8FM_AI
!git pull --ff-only || true
!ls

## Запуск обучения

Драйвер сам посмотрит на `driver_state.json` и, если он есть, продолжит
обучение с прошлого шага. Меняйте `--total-updates` под свои цели
(больше — дольше и сильнее). `--time-budget-seconds` останавливает работу
до лимита сессии, чтобы Kaggle успел сохранить результат.

Первый запуск делает длинный BC-разогрев (~10–15 мин на 6000 игр), дальше
идут чанки PPO. Прогресс печатается по каждому чанку.

In [ ]:
!python kaggle/train_lu_sol.py \
    --work-dir {WORK_DIR} \
    --workers 4 \
    --total-updates 6000 \
    --chunk-updates 200 \
    --games-per-update 48 \
    --warm-start-games 6000 \
    --time-budget-seconds 30000

## Возобновление в следующей сессии

Просто перезапустите ноутбук заново (**Save & Run All**): драйвер сам найдёт
`driver_state.json` в `/kaggle/working` и продолжит с того же шага, сохраняя
единое монотонное расписание learning rate и энтропии.

In [ ]:
import json, os

state_path = os.path.join(WORK_DIR, "driver_state.json")
if os.path.exists(state_path):
    print(open(state_path, encoding="utf-8").read())

for name in (
    "lu_sol.json",
    "lu_sol-latest.json",
    "lu_sol-metrics.json",
):
    path = os.path.join(WORK_DIR, name)
    if os.path.exists(path):
        print(f"{name:34} {os.path.getsize(path)/1e6:6.1f} MB")

## Забрать модель на ноутбук

После завершения сессии откройте вкладку **Output** этого ноутбука справа
и скачайте:

* `lu_sol.json` — лучшая модель (её и подкладывайте в проект);
* `lu_sol-latest.json` — последняя модель (нужна только для продолжения
  обучения, на ноутбуке не обязательна);
* `lu_sol-metrics.json` — метрики и кривые для графиков.

Положите `lu_sol.json` в папку `models/` локального проекта и укажите путь
в настройках приложения.

> Внимание: расклад признаков `v2` **не совместим** со старым `lu_luna`
> (он на `v1`), поэтому продолжайте обучение `lu_sol` только из
> `lu_sol`-чекпоинтов. Старая модель продолжает работать как есть.